# Beginner E1: classify accents with Librosa and Logistic Regression

This is a small, step-by-step learning version of the E1 experiment. It avoids corpus download code so you can focus on the machine-learning workflow.

By the end, you will know how to:

1. load audio with **Librosa**;
2. turn each recording into a fixed-length feature vector;
3. keep speakers separate between training and testing;
4. train a scaled **Logistic Regression** classifier; and
5. read F1, balanced accuracy and a confusion matrix.

> This experiment distinguishes the two supplied speech datasets. It must not be used to infer ethnicity, nationality or identity.

In [ ]:
# Install the libraries used in this notebook.
%pip -q install "librosa>=0.10.2" scikit-learn seaborn joblib soundfile

## 1. Arrange the audio by class and speaker

Create one `Singlish` project folder in Google Drive and manually upload the audio into `data/audio`. The notebook will place its generated files in `outputs/e1`:

```text
Singlish/
├── data/
│   └── audio/
│       ├── singapore/
│       │   ├── speaker_001/
│       │   │   ├── clip_01.wav
│       │   │   └── clip_02.wav
│       │   └── speaker_002/
│       │       └── clip_01.wav
│       └── non_singapore/
│           ├── speaker_101/
│           │   └── clip_01.mp3
│           └── speaker_102/
│               └── clip_01.mp3
```

Only `data/audio` needs to exist before the run. The notebook automatically creates `Singlish/outputs/e1` for its results.

The first directory below a class must identify the speaker. Each class needs at least two speakers; five or more per class is preferable. Speaker folders are essential because clips from one speaker must never occur in both training and test data.

For this project, `singapore` should contain MNSC clips and `non_singapore` should contain the selected Common Voice comparison accents. Start with a small subset while learning.

In [ ]:
from pathlib import Path

# Google Colab: mount Drive.
USE_GOOGLE_DRIVE = True
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')

# Configure only this path.
PROJECT_ROOT = Path('/content/drive/MyDrive/Singlish')
DATA_ROOT = PROJECT_ROOT / 'data' / 'audio'
OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'e1'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 42
SAMPLE_RATE = 16_000
MAX_SECONDS = 6

print('Project folder:', PROJECT_ROOT)
print('Looking for audio in:', DATA_ROOT)
print('Saving results to:', OUTPUT_DIR)

In [ ]:
import json
from collections import Counter

import joblib
import librosa
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, classification_report,
    confusion_matrix, f1_score,
)
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

np.random.seed(RANDOM_SEED)
print('Librosa version:', librosa.__version__)

## 2. Find the recordings

We give Singapore speech label `1` and non-Singapore speech label `0`. A label is the answer the classifier learns to predict.

In [ ]:
AUDIO_EXTENSIONS = {'.wav', '.mp3', '.flac', '.ogg', '.m4a'}
CLASS_LABELS = {'non_singapore': 0, 'singapore': 1}


def find_recordings(data_root):
    rows = []
    for class_name, label in CLASS_LABELS.items():
        class_dir = data_root / class_name
        if not class_dir.is_dir():
            raise FileNotFoundError(f'Missing class directory: {class_dir}')
        for path in sorted(class_dir.rglob('*')):
            if not path.is_file() or path.suffix.lower() not in AUDIO_EXTENSIONS:
                continue
            relative_parts = path.relative_to(class_dir).parts
            if len(relative_parts) < 2:
                raise ValueError(f'Put this clip inside a speaker folder: {path}')
            speaker_id = relative_parts[0]
            # Prefixing avoids accidental ID collisions between the two corpora.
            group_id = f'{class_name}::{speaker_id}'
            rows.append((path, label, group_id))
    if not rows:
        raise RuntimeError(f'No supported audio files found below {data_root}')
    return rows


recordings = find_recordings(DATA_ROOT)
print('Recordings:', len(recordings))
print('Labels:', Counter(label for _, label, _ in recordings))
print('Speakers per class:')
for class_name, label in CLASS_LABELS.items():
    speakers = {group for _, item_label, group in recordings if item_label == label}
    print(f'  {class_name}: {len(speakers)}')

## 3. Convert audio into numbers

Logistic Regression cannot read a waveform directly. Librosa measures useful acoustic properties over many short frames. We summarize each property with its mean and standard deviation so every clip becomes one vector of 39 numbers.

The features are:

- **MFCCs:** a compact description of spectral shape;
- **F0:** an estimate of voice pitch;
- **spectral centroid, bandwidth and roll-off:** different summaries of frequency content;
- **zero-crossing rate:** how often the signal changes sign; and
- **RMS energy:** signal strength over time.

In [ ]:
N_MFCC = 13
N_FFT = 1024
HOP_LENGTH = 256


def load_and_prepare_audio(path):
    # sr=16000 resamples every file consistently; mono=True merges channels.
    y, _ = librosa.load(path, sr=SAMPLE_RATE, mono=True)
    y, _ = librosa.effects.trim(y, top_db=30)
    if y.size == 0:
        raise ValueError(f'No audible signal: {path}')
    peak = np.max(np.abs(y))
    if peak > 0:
        y = y / peak
    # One second is enough for stable frame statistics; long clips are capped.
    y = librosa.util.fix_length(y, size=max(len(y), SAMPLE_RATE))
    return y[:SAMPLE_RATE * MAX_SECONDS].astype(np.float32)


def mean_and_std(feature):
    return np.concatenate([np.mean(feature, axis=1), np.std(feature, axis=1)])


def extract_features(path):
    y = load_and_prepare_audio(path)
    mfcc = librosa.feature.mfcc(
        y=y, sr=SAMPLE_RATE, n_mfcc=N_MFCC,
        n_fft=N_FFT, hop_length=HOP_LENGTH,
    )
    rms = librosa.feature.rms(y=y, frame_length=N_FFT, hop_length=HOP_LENGTH)
    f0 = librosa.yin(
        y, fmin=librosa.note_to_hz('C2'), fmax=librosa.note_to_hz('C7'),
        sr=SAMPLE_RATE, frame_length=2048, hop_length=HOP_LENGTH,
    )
    # Ignore very quiet frames when summarizing pitch.
    energy = rms.ravel()[:len(f0)]
    voiced = np.isfinite(f0) & (energy > max(float(np.median(energy)) * 0.1, 1e-4))
    voiced_f0 = f0[voiced]
    pitch_stats = [
        float(np.mean(voiced_f0)) if voiced_f0.size else 0.0,
        float(np.std(voiced_f0)) if voiced_f0.size else 0.0,
        float(np.mean(voiced)),
    ]
    time_features = [
        librosa.feature.spectral_centroid(y=y, sr=SAMPLE_RATE, n_fft=N_FFT, hop_length=HOP_LENGTH),
        librosa.feature.spectral_bandwidth(y=y, sr=SAMPLE_RATE, n_fft=N_FFT, hop_length=HOP_LENGTH),
        librosa.feature.spectral_rolloff(y=y, sr=SAMPLE_RATE, n_fft=N_FFT, hop_length=HOP_LENGTH),
        librosa.feature.zero_crossing_rate(y, frame_length=N_FFT, hop_length=HOP_LENGTH),
        rms,
    ]
    vector = np.concatenate([mean_and_std(mfcc), pitch_stats] + [mean_and_std(x) for x in time_features])
    vector = np.nan_to_num(vector, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)
    assert vector.shape == (39,)
    return vector

In [ ]:
feature_rows, kept_rows = [], []
for number, row in enumerate(recordings, start=1):
    path, label, group_id = row
    try:
        feature_rows.append(extract_features(path))
        kept_rows.append(row)
    except Exception as error:
        print(f'Skipping {path.name}: {error}')
    if number % 25 == 0 or number == len(recordings):
        print(f'Processed {number}/{len(recordings)}')

X = np.vstack(feature_rows)
y = np.asarray([label for _, label, _ in kept_rows], dtype=np.int64)
groups = np.asarray([group for _, _, group in kept_rows])
paths = np.asarray([str(path) for path, _, _ in kept_rows])

assert X.shape[1] == 39 and np.isfinite(X).all()
print('Feature matrix shape:', X.shape)
print('One row = one recording; one column = one acoustic statistic.')

## 4. Make a speaker-disjoint split

A random clip split can put the same voice in training and testing. The model might then recognize speakers instead of learning general accent patterns. `StratifiedGroupKFold` keeps each speaker in exactly one side while trying to preserve both labels.

With at least five speakers per class, the first fold gives an approximately 80/20 training/test split. With fewer speakers, the notebook automatically uses fewer folds.

In [ ]:
speaker_counts = {
    label: len(set(groups[y == label]))
    for label in sorted(np.unique(y))
}
if set(speaker_counts) != {0, 1}:
    raise ValueError('Both class labels must remain after feature extraction')
n_splits = min(5, min(speaker_counts.values()))
if n_splits < 2:
    raise ValueError('Each class needs audio from at least two speakers')

splitter = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_SEED)
train_idx, test_idx = next(splitter.split(X, y, groups))

train_speakers = set(groups[train_idx])
test_speakers = set(groups[test_idx])
assert not (train_speakers & test_speakers)
assert set(np.unique(y[train_idx])) == {0, 1}
assert set(np.unique(y[test_idx])) == {0, 1}

print('Training clips:', len(train_idx), '| labels:', Counter(y[train_idx]))
print('Test clips:    ', len(test_idx), '| labels:', Counter(y[test_idx]))
print('Speaker overlap:', len(train_speakers & test_speakers))

## 5. Train Logistic Regression

`StandardScaler` puts differently sized measurements on comparable scales. This matters for Logistic Regression because large-valued features could otherwise dominate the fitted coefficients. Logistic Regression learns a weighted linear boundary and converts its score into a class probability.

The scaler and classifier are joined in one pipeline, which prevents test-set information from leaking into scaling. `class_weight='balanced'` gives each class equal importance when the clip counts differ.

In [ ]:
model = Pipeline([
    ('scale', StandardScaler()),
    ('logistic_regression', LogisticRegression(
        C=1.0, class_weight='balanced', max_iter=2_000,
        random_state=RANDOM_SEED,
    )),
])
model.fit(X[train_idx], y[train_idx])
print('Training complete.')

## 6. Evaluate unseen speakers

Useful metrics:

- **Balanced accuracy** averages recall across both classes, so each class matters equally.
- **Macro F1** calculates F1 for each class and gives the two classes equal weight.
- A **confusion matrix** shows exactly which kinds of mistakes were made. Rows are true labels and columns are predictions.

In [ ]:
predictions = model.predict(X[test_idx])
matrix = confusion_matrix(y[test_idx], predictions, labels=[0, 1])
metrics = {
    'accuracy': float(accuracy_score(y[test_idx], predictions)),
    'balanced_accuracy': float(balanced_accuracy_score(y[test_idx], predictions)),
    'macro_f1': float(f1_score(y[test_idx], predictions, average='macro')),
    'confusion_matrix': matrix.tolist(),
}
print(json.dumps(metrics, indent=2))
print('\nDetailed report:\n')
print(classification_report(
    y[test_idx], predictions, labels=[0, 1],
    target_names=['Non-Singapore', 'Singapore'], digits=3, zero_division=0,
))

In [ ]:
sns.set_theme(style='white')
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(
    matrix, annot=True, fmt='d', cmap='Blues', cbar=False, ax=ax,
    xticklabels=['Non-SG', 'Singapore'], yticklabels=['Non-SG', 'Singapore'],
)
ax.set(title='Beginner E1 Logistic Regression test', xlabel='Predicted label', ylabel='True label')
fig.tight_layout()
confusion_path = OUTPUT_DIR / 'logistic_regression_confusion_matrix.png'
fig.savefig(confusion_path, dpi=180, bbox_inches='tight')
plt.show()

## 7. Save the experiment

Saving the model and exact split makes the result reproducible. The feature cache also lets you try SVM or Random Forest later without extracting the audio features again. All files below are saved inside `Singlish/outputs/e1` in Google Drive.

In [ ]:
model_path = OUTPUT_DIR / 'beginner_librosa_logistic_regression.joblib'
cache_path = OUTPUT_DIR / 'beginner_acoustic_features.npz'
report_path = OUTPUT_DIR / 'beginner_logistic_regression_report.json'

joblib.dump(model, model_path)
np.savez_compressed(
    cache_path, features=X, labels=y, groups=groups, paths=paths,
    train_indices=train_idx, test_indices=test_idx,
)
report = {
    'model': 'StandardScaler + LogisticRegression',
    'label_meaning': {'0': 'non_singapore', '1': 'singapore'},
    'feature_count': int(X.shape[1]),
    'train_samples': int(len(train_idx)),
    'test_samples': int(len(test_idx)),
    'train_speakers': int(len(train_speakers)),
    'test_speakers': int(len(test_speakers)),
    'speaker_overlap': 0,
    'metrics': metrics,
    'caution': 'Corpus and recording differences may be learned instead of accent.',
}
report_path.write_text(json.dumps(report, indent=2) + '\n')

print('Saved:')
for path in (model_path, cache_path, report_path, confusion_path):
    print(' -', path)

## 8. Try one recording

This final helper applies the exact same feature extraction and fitted pipeline to one file. Treat the probability as a model score, not a definitive statement about a person.

In [ ]:
def predict_recording(audio_path):
    features = extract_features(Path(audio_path)).reshape(1, -1)
    probability = float(model.predict_proba(features)[0, 1])
    predicted_name = 'Singapore' if probability >= 0.5 else 'Non-Singapore'
    print(f'Prediction: {predicted_name}')
    print(f'Singapore-class probability: {probability:.1%}')
    return probability

# Example — replace this with one of your audio paths:
# predict_recording(DATA_ROOT / 'singapore' / 'speaker_001' / 'clip_01.wav')

## What to try next

Once this notebook makes sense:

1. increase the number and variety of speakers;
2. add a separate validation split before tuning Logistic Regression settings;
3. compare this result with SVM and Random Forest using the same feature cache and split; and
4. investigate whether corpus, microphone or prompt differences explain unusually high results.

Do not repeatedly adjust the model after looking at the test score. The full notebook uses a separate validation set for model selection.